# GDELT News Headlines — WTI Crude Oil

This notebook queries the GDELT Project API to retrieve news article headlines 
related to WTI crude oil prices over a two-year period (2024–2026). Articles are 
downloaded week by week to avoid API rate limiting. The raw dataset is then cleaned 
by filtering to English-language articles only, removing duplicates, and parsing 
timestamps. The cleaned dataset contains article titles, publication timestamps, 
source domains, and URLs, and is saved to 
`01_data/processed/gdelt_wti_clean.csv`.

### General imports

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

### Main search function

Adjusts request headers so that instead of identifying as Python, requests are sent with a Mozilla browser User-Agent string. This avoids being blocked by GDELT's bot detection.

**Note:** GDELT API results are capped at 250 articles per request. Weeks where the download returns exactly 250 articles may be truncated. This affects a small number of high-activity periods and is considered acceptable for the purposes of this study.

In [2]:
def search_gdelt(query, start_date, end_date, max_records=250):
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query": query,
        "mode": "artlist",
        "maxrecords": max_records,
        "startdatetime": start_date.strftime("%Y%m%d%H%M%S"),
        "enddatetime": end_date.strftime("%Y%m%d%H%M%S"),
        "format": "json"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers, timeout=30)
    return response.json()

### Download by date range

Executes the search function iteratively over a range of weeks between `start_date` and `end_date`. A cooldown between requests is included to reduce the number of rejections caused by GDELT's rate limiting.

In [3]:
def download_gdelt_range(query, start_date, end_date, sleep_seconds=8):
    all_articles = []
    current = start_date
    
    while current < end_date:
        week_end = min(current + timedelta(days=7), end_date)
        try:
            results = search_gdelt(query, current, week_end)
            articles = results.get("articles", [])
            for a in articles:
                a['week_start'] = current.strftime("%Y-%m-%d")
            all_articles.extend(articles)
            print(f"{current.date()} → {week_end.date()}: {len(articles)} articles")
        except Exception as e:
            print(f"{current.date()} → {week_end.date()}: ERROR — {str(e)[:50]}")
        
        current = week_end
        time.sleep(sleep_seconds)
    
    return pd.DataFrame(all_articles)

### Download weekly headlines

GDELT does not provide article body text through its API — only metadata including title, URL, publication date, domain, and language. Body text is retrieved separately in `04_gdelt_scraper.ipynb` by scraping each article URL individually.

In [5]:
# Celda — Additional queries
additional_queries = [
    "crude oil market",
    "OPEC production cut",
    "oil supply demand",
    "petroleum inventory",
    "Brent crude price",
    "oil inventory report",
    "energy market outlook",
    "crude oil WTI price",
]

dfs = []  # start with existing results

for query in additional_queries:
    print(f"\nDownloading: '{query}'")
    df_query = download_gdelt_range(
        query=query,
        start_date=datetime(2024, 1, 1),
        end_date=datetime(2026, 3, 1)
    )
    print(f"  → {len(df_query)} articles")
    dfs.append(df_query)

# Combine all results
df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal before deduplication: {len(df_all)}")


Downloading: 'crude oil market'
2024-01-01 → 2024-01-08: 250 articles
2024-01-08 → 2024-01-15: 250 articles
2024-01-15 → 2024-01-22: 250 articles
2024-01-22 → 2024-01-29: 250 articles
2024-01-29 → 2024-02-05: ERROR — Expecting value: line 1 column 1 (char 0)
2024-02-05 → 2024-02-12: ERROR — Expecting value: line 1 column 1 (char 0)
2024-02-12 → 2024-02-19: 250 articles
2024-02-19 → 2024-02-26: ERROR — Expecting value: line 1 column 1 (char 0)
2024-02-26 → 2024-03-04: ERROR — Expecting value: line 1 column 1 (char 0)
2024-03-04 → 2024-03-11: ERROR — Expecting value: line 1 column 1 (char 0)
2024-03-11 → 2024-03-18: 250 articles
2024-03-18 → 2024-03-25: 250 articles
2024-03-25 → 2024-04-01: 250 articles
2024-04-01 → 2024-04-08: 250 articles
2024-04-08 → 2024-04-15: 250 articles
2024-04-15 → 2024-04-22: 250 articles
2024-04-22 → 2024-04-29: 250 articles
2024-04-29 → 2024-05-06: 250 articles
2024-05-06 → 2024-05-13: 250 articles
2024-05-13 → 2024-05-20: 250 articles
2024-05-20 → 2024-05-2

In [6]:
# Deduplicate and save raw

df_all = df_all.drop_duplicates(subset='url').reset_index(drop=True)
print(f"Total after deduplication: {len(df_all)}")
print(f"\nLanguages:\n{df_all['language'].value_counts().head()}")
print(f"\nTop domains:\n{df_all['domain'].value_counts().head(10)}")

filename_raw = "gdelt_wti_raw.csv"
df_all.to_csv(f"../01_data/raw/news/{filename_raw}", index=False)
print(f"\nSaved to 01_data/raw/news/{filename_raw}")

Total after deduplication: 51948

Languages:
language
English       19552
Chinese       19262
Vietnamese     2163
Arabic         1894
Spanish        1314
Name: count, dtype: int64

Top domains:
domain
marketscreener.com       1560
finance.eastmoney.com    1466
mp.cnfol.com             1447
gold.cnfol.com           1293
forex.cnfol.com          1146
163.com                   791
finance.sina.com.cn       754
chinatimes.com            732
futures.cnfol.com         724
udn.com                   686
Name: count, dtype: int64

Saved to 01_data/raw/news/gdelt_wti_raw.csv


### Find failed weeks

Where not a single query got news from specific week "x"

In [7]:
# Celda — Find weeks with zero coverage across all queries

df_existing = pd.read_csv("../01_data/raw/news/gdelt_wti_raw.csv")
df_existing['week_start'] = pd.to_datetime(df_existing['week_start'])

# Build full list of expected weeks
start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 3, 1)

all_weeks = []
current = start_date
while current < end_date:
    week_end = min(current + timedelta(days=7), end_date)
    all_weeks.append((current, week_end))
    current = week_end

# Find weeks with zero articles across ALL queries
downloaded_weeks = set(df_existing['week_start'].dt.date)
truly_empty = [
    (start, end) for start, end in all_weeks
    if start.date() not in downloaded_weeks
]

print(f"Total expected weeks: {len(all_weeks)}")
print(f"Weeks with at least one article: {len(downloaded_weeks)}")
print(f"Weeks with zero coverage across all queries: {len(truly_empty)}")

if truly_empty:
    print("\nEmpty weeks:")
    for start, end in truly_empty:
        print(f"  {start.date()} → {end.date()}")
else:
    print("\nAll weeks have at least some coverage.")

Total expected weeks: 113
Weeks with at least one article: 106
Weeks with zero coverage across all queries: 7

Empty weeks:
  2025-02-17 → 2025-02-24
  2025-03-31 → 2025-04-07
  2025-06-16 → 2025-06-23
  2025-06-23 → 2025-06-30
  2025-06-30 → 2025-07-07
  2025-10-20 → 2025-10-27
  2026-02-16 → 2026-02-23


### Summary

In [8]:
print(f"Total articles: {len(df_all)}")
print(f"Weeks covered: {df_all['week_start'].nunique()}")
print(f"\nLanguages:\n{df_all['language'].value_counts().head()}")
print(f"\nTop domains:\n{df_all['domain'].value_counts().head(10)}")
print(df_all[['title', 'seendate', 'domain']].head(5))

Total articles: 51948
Weeks covered: 106

Languages:
language
English       19552
Chinese       19262
Vietnamese     2163
Arabic         1894
Spanish        1314
Name: count, dtype: int64

Top domains:
domain
marketscreener.com       1560
finance.eastmoney.com    1466
mp.cnfol.com             1447
gold.cnfol.com           1293
forex.cnfol.com          1146
163.com                   791
finance.sina.com.cn       754
chinatimes.com            732
futures.cnfol.com         724
udn.com                   686
Name: count, dtype: int64
                                               title          seendate  \
0                 2024年首轮成品油价格调整 汽柴油市场呈现分化格局 _ 东方财富网  20240103T223000Z   
1  原油周评 ： 新年第一周原油收高 ， 中东局势主导价格 _ 国际原油市场 _ 黄金网 _ 中金在线  20240106T083000Z   
2  刘铭诚 ： 1 . 4黄金暴跌 # 原油暴涨行情解析 ， 今日黄金原油操作建议 _ 国际原油...  20240104T100000Z   
3  主次节奏 ： 原油趋势向下 ， 日内测试68 . 50 _ 国际原油市场 _ 黄金网 _ 中金在线  20240103T043000Z   
4                      2024年首次调价 ！ 国内成品油价格将结束  六连跌    20240103T081500Z   

                  do

### News cleaning

The NLP models used in this project (FinBERT) are trained primarily on English financial text. Non-English articles are therefore excluded to maximize sentiment classification accuracy. The following steps are applied:

- **Language filter:** Only English-language articles are retained.
- **Deduplication:** Articles with identical titles are removed, keeping the first occurrence.
- **Timestamp parsing:** The `seendate` field is converted from GDELT's raw format (`YYYYMMDDTHHmmSSZ`) to a standard UTC datetime.
- **Column selection:** Only the columns relevant for downstream modeling are kept: `title`, `datetime`, `domain`, and `url`.

In [ ]:
df_clean = pd.read_csv("../01_data/raw/news/gdelt_wti_raw.csv")

# Keep English articles only
df_clean = df_clean[df_clean['language'] == 'English']
print(f"Articles after filtering by english: {len(df_clean)}")

# Remove duplicate titles
df_clean = df_clean.drop_duplicates(subset='title')
print(f"Articles after removing duplicates: {len(df_clean)}")

# Parse publication timestamp to UTC datetime
df_clean['datetime'] = pd.to_datetime(df_clean['seendate'], format='%Y%m%dT%H%M%SZ', utc=True)
df_clean = df_clean.sort_values('datetime').reset_index(drop=True)

# Keep only relevant columns
df_clean = df_clean[['title', 'datetime', 'domain', 'url']]

print(f"Range: {df_clean['datetime'].min()} to {df_clean['datetime'].max()}")
print(df_clean.head(5))

filename = "gdelt_wti_clean.csv"
df_clean.to_csv(f"../01_data/processed/{filename}", index=False)
print(f"Saved to 01_data/processed/{filename}")

News after filtering by english: 19552
News after removing duplicates: 16326
Articles after cleaning: 16326
Range: 2024-01-01 00:30:00+00:00 to 2026-03-01 00:30:00+00:00
                                               title  \
0                2023 Was a Bad Year for Commodities   
1  Will oil fare better in 2024 ? - BR Research -...   
2  Global oil market  poised for significant shif...   
3  News Highlights : Top Energy News of the Day -...   
4  Oil prices shed 10 % in 2023 as supply , deman...   

                   datetime              domain  \
0 2024-01-01 00:30:00+00:00        oilprice.com   
1 2024-01-01 04:45:00+00:00       brecorder.com   
2 2024-01-01 05:00:00+00:00           zawya.com   
3 2024-01-01 06:30:00+00:00  marketscreener.com   
4 2024-01-01 09:30:00+00:00           zawya.com   

                                                 url  
0  https://oilprice.com/Energy/Energy-General/202...  
1  https://www.brecorder.com/news/40281468/will-o...  
2  https://www.zawya.